In [3]:
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from uuid import uuid4

# Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [4]:
docs = [
    Document(page_content="Supervised learning trains models on labeled input-output pairs to predict unseen data.", metadata={"topic": "ml"}),
    Document(page_content="Neural networks learn by adjusting weights through backpropagation and gradient descent.", metadata={"topic": "ml"}),
    Document(page_content="Overfitting occurs when a model memorises training data and performs poorly on new data.", metadata={"topic": "ml"}),
    Document(page_content="Training data quality and quantity are the most important factors in model performance.", metadata={"topic": "ml"}),
    Document(page_content="Cross-validation splits data into folds to evaluate model generalisation more reliably.", metadata={"topic": "ml"}),
    Document(page_content="Inflation is the rate at which the general level of prices for goods and services rises over time.", metadata={"topic": "economics"}),
    Document(page_content="GDP measures the total monetary value of all goods and services produced within a country.", metadata={"topic": "economics"}),
    Document(page_content="Interest rates set by central banks influence borrowing costs and consumer spending.", metadata={"topic": "economics"}),
    Document(page_content="Caramelisation occurs when sugar is heated above 160°C, creating complex flavour compounds.", metadata={"topic": "cooking"}),
    Document(page_content="Fermentation uses microorganisms to convert sugars into alcohol or acids, preserving food.", metadata={"topic": "cooking"}),
    Document(page_content="A marathon is a long-distance race of exactly 42.195 kilometres, run on roads.", metadata={"topic": "sports"}),
    Document(page_content="Tennis scoring follows a love, 15, 30, 40, game sequence, with deuce at 40-40.", metadata={"topic": "sports"}),
]

In [5]:
vectorstore = Chroma(
    embedding_function=embeddings,
    collection_name="demo",
    collection_configuration={
        "hnsw": {
            "space": 'cosine'      # cosine, L2, ip
        }
    }
)

In [6]:
# add documents to vector store

ids_added = vectorstore.add_documents(documents=docs,
                                      ids=[str(uuid4()) for _ in range(len(docs))])

In [12]:
query = "How does ML model training work?"

similarity_scores = vectorstore.similarity_search_with_score(query, k=3)


In [13]:
# get all the scores in (1-score) format
# actual cosine similarity scores
for _, score in similarity_scores:
    print(f"Similarity Score: {1 - score:.4f}")

Similarity Score: 0.6951
Similarity Score: 0.6922
Similarity Score: 0.6748


In [8]:
similarity_scores

[(Document(id='8a0c7db8-c6a0-4065-b1bf-fdae1b4b3ec9', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
  0.30493390560150146),
 (Document(id='695bc2e6-5ca5-4544-bdc2-74f8aa08e59e', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  0.30777978897094727),
 (Document(id='2c0b1310-8f4c-45ec-abe7-905833f994c4', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
  0.3252291679382324)]

In [10]:
query = "How does ML model training work?"

threshold = 0.66

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": threshold},
)

results = retriever.invoke(query)
results

[Document(id='4d04747a-7d62-4b41-a715-040f894ccddd', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
 Document(id='21a5e48f-9f6a-4fa3-b7aa-2f0941b8d38b', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
 Document(id='c3218d48-fb79-495f-85f9-242f9d75d26c', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.')]